Option 3: Response Time Disparities
Do response times for 311 requests differ across neighborhoods or complaint types? Calculate the time between request creation and closure, and examine whether there are systematic differences by location or complaint category.

Loading the Data

In [2]:
%pip install matplotlib seaborn scipy

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 38.5 MB/s  0:00:00 eta 0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 39.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 39.6 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [seaborn]m7/8 [seaborn]ib]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. Retrieve Data using Socrata API
# We filter for 2024 data and limit to 50,000 rows for performance.
query_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=50000&$where=created_date>'2024-01-01T00:00:00'"
df = pd.read_csv(query_url)

# 2. Initial Exploration
print(f"Dataset Shape: {df.shape}")
print(df[['created_date', 'closed_date', 'complaint_type', 'borough']].head())

/var/folders/l6/mdqsw8wx0hq7jzj0bwmfdf640000gn/T/ipykernel_13203/117305549.py:10: DtypeWarning: Columns (0: taxi_company_borough) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(query_url)


Dataset Shape: (50000, 44)
              created_date closed_date       complaint_type   borough
0  2026-02-23T02:03:45.000         NaN          Snow or Ice  BROOKLYN
1  2026-02-23T02:02:39.000         NaN         Damaged Tree  BROOKLYN
2  2026-02-23T02:02:07.000         NaN     Blocked Driveway     BRONX
3  2026-02-23T02:01:16.000         NaN  Noise - Residential  BROOKLYN
4  2026-02-23T02:00:18.000         NaN      Illegal Parking  BROOKLYN


Data Cleaning

In [6]:
# Convert to datetime
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

# Drop rows where 'closed_date' is missing (request still open)
df = df.dropna(subset=['closed_date'])

# Create derived variable: Response Time in Hours
df['response_time_hrs'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600

# Remove outliers/errors: negative times or extremely long times (> 30 days)
df = df[(df['response_time_hrs'] > 0) & (df['response_time_hrs'] < 720)]

print(f"Cleaned Dataset Shape: {df.shape}")

Cleaned Dataset Shape: (32185, 45)
